## 1. Confirm all four Gold tables exist

In [0]:
%sql
SHOW TABLES IN credit_card_project.gold;


expected tables:
- gold_user_behavior
- gold_card_behavior
- gold_merchant_behavior
- gold_fraud_features


## 2. Confirm Gold row counts

In [0]:
%sql
SELECT
    'gold_user_behavior' AS table_name,
    COUNT(*) AS row_count
FROM credit_card_project.gold.gold_user_behavior --expected: 2,000

UNION ALL

SELECT
    'gold_card_behavior',
    COUNT(*)
FROM credit_card_project.gold.gold_card_behavior --expected: 6,146

UNION ALL

SELECT
    'gold_merchant_behavior',
    COUNT(*)
FROM credit_card_project.gold.gold_merchant_behavior --expected: 100,343

UNION ALL

SELECT
    'gold_fraud_features',
    COUNT(*)
FROM credit_card_project.gold.gold_fraud_features; --expected: 24,386,900


## 3. Confirm Silver → Fraud Features transaction population

In [0]:
%sql
-- expected: gold_fraud_features is supposed to be one row per Silver transaction
SELECT *, CASE WHEN silver_rows = gold_fraud_rows THEN 'PASS' ELSE 'FAIL' END AS result 
FROM (
    SELECT
        (SELECT COUNT(*) FROM credit_card_project.silver.cleaned_transactions) AS silver_rows,
        (SELECT COUNT(*) FROM credit_card_project.gold.gold_fraud_features) AS gold_fraud_rows);


In [0]:
%sql
DESCRIBE credit_card_project.silver.cleaned_transactions;


In [0]:
%sql
DESCRIBE credit_card_project.gold.gold_fraud_features;


## 4. Amount and fraud reconciliation

In [0]:
%sql
SELECT
    s.row_count AS silver_rows,
    g.row_count AS gold_rows,
    s.total_amount AS silver_total_amount,
    g.total_amount AS gold_total_amount,
    s.fraud_count AS silver_fraud_count,
    g.fraud_count AS gold_fraud_count,
    ROUND(g.total_amount - s.total_amount, 2) AS amount_difference,
    g.fraud_count - s.fraud_count AS fraud_difference
FROM
(
    SELECT
        COUNT(*) AS row_count,
        SUM(amount) AS total_amount,
        SUM(CASE WHEN is_fraud THEN 1 ELSE 0 END) AS fraud_count
    FROM credit_card_project.silver.cleaned_transactions
) s
CROSS JOIN
(
    SELECT
        COUNT(*) AS row_count,
        SUM(amount) AS total_amount,
        SUM(CASE WHEN is_fraud THEN 1 ELSE 0 END) AS fraud_count
    FROM credit_card_project.gold.gold_fraud_features
) g;


/*
expected:
silver_rows          24,386,900
gold_rows            24,386,900
silver_total_amount  1,064,098,117.16
gold_total_amount    1,064,098,117.16
silver_fraud_count   29,757
gold_fraud_count     29,757
amount_difference    0.00
fraud_difference     0
*/

## 5. Validate the aggregate Gold grains

User grain

In [0]:
%sql
--confirm that the declared business keys are actually unique.
SELECT
    COUNT(*) AS rows,
    COUNT(DISTINCT user_id) AS distinct_users,
    COUNT(*) - COUNT(DISTINCT user_id) AS duplicate_rows
FROM credit_card_project.gold.gold_user_behavior;
/*
expected:
rows             2000
distinct_users   2000
duplicate_rows   0
*/


Card grain

In [0]:
%sql
SELECT
    COUNT(*) AS rows,
    COUNT(DISTINCT STRUCT(user_id, card_id)) AS distinct_card_keys,
    COUNT(*) - COUNT(DISTINCT STRUCT(user_id, card_id)) AS duplicate_rows
FROM credit_card_project.gold.gold_card_behavior;
/*
expected:
rows                6146
distinct_card_keys  6146
duplicate_rows      0
*/


Merchant grain

In [0]:
%sql
SELECT
    COUNT(*) AS rows,
    COUNT(DISTINCT merchant_id) AS distinct_merchants,
    COUNT(*) - COUNT(DISTINCT merchant_id) AS duplicate_rows
FROM credit_card_project.gold.gold_merchant_behavior;
/*
expected:
rows                 100343
distinct_merchants   100343
duplicate_rows       0
*/


## 6. Critical NULL checks

In [0]:
%sql
-- focus: keys and core transaction fields
SELECT
    'gold_user_behavior' AS table_name,
    SUM(CASE WHEN user_id IS NULL THEN 1 ELSE 0 END) AS null_user_id
FROM credit_card_project.gold.gold_user_behavior

UNION ALL

SELECT
    'gold_card_behavior',
    SUM(CASE WHEN user_id IS NULL OR card_id IS NULL THEN 1 ELSE 0 END)
FROM credit_card_project.gold.gold_card_behavior

UNION ALL

SELECT
    'gold_merchant_behavior',
    SUM(CASE WHEN merchant_id IS NULL THEN 1 ELSE 0 END)
FROM credit_card_project.gold.gold_merchant_behavior

UNION ALL

SELECT
    'gold_fraud_features',
    SUM(
        CASE
            WHEN user_id IS NULL
              OR card_id IS NULL
              OR transaction_timestamp IS NULL
              OR amount IS NULL
              OR merchant_id IS NULL
              OR is_fraud IS NULL
            THEN 1
            ELSE 0
        END
    )
FROM credit_card_project.gold.gold_fraud_features;


## 7. Final persisted-table sanity check

expected: format = delta

In [0]:
%sql
DESCRIBE DETAIL credit_card_project.gold.gold_user_behavior;

In [0]:
%sql
DESCRIBE DETAIL credit_card_project.gold.gold_card_behavior;

In [0]:
%sql
DESCRIBE DETAIL credit_card_project.gold.gold_merchant_behavior;

In [0]:
%sql
DESCRIBE DETAIL credit_card_project.gold.gold_fraud_features;